# Universe Coverage (Slice 6A)

OwnerLens's SEC-first canonical pipeline was validated on a handful of companies. Before designing the next normalization improvement, this notebook measures how it holds up across **24 conventional U.S. operating companies** in several industries:

```
SEC fetch → canonical history → owner economics → capital efficiency → economic value
          → compounding → capital allocation → summary → coverage
```

This is **coverage discovery**. Nothing here changes SEC normalization or adds ticker overrides. Every blocker is diagnosed from the raw Company Facts using a small read-only catalog of alternative SEC concepts.

The survey was run with `uv run python scripts/survey_universe.py`, which ingests through the existing `ingest_company` into a dedicated `data/universe_6a.db`. This notebook replays it **offline** from the stored raw snapshots, so it is reproducible and makes no SEC requests.

In [1]:
import json
from pathlib import Path

from owner_lens.persistence import FilesystemRawSnapshotStore
from owner_lens.universe import (
    UNIVERSE_6A,
    UniverseReport,
    format_company_view,
    format_metric_view,
    format_pattern_view,
    survey_from_snapshot,
)
from owner_lens.universe import DiagnosisCategory as D
from owner_lens.universe import RestatementKind as R

repo_root = next((p for p in (Path('.'), Path('..')) if (p / 'pyproject.toml').exists()), Path('.'))
manifest = json.loads((repo_root / 'data' / 'universe_6a_report.json').read_text(encoding='utf-8'))
raw_store = FilesystemRawSnapshotStore(repo_root / 'data' / 'raw' / 'sec' / 'company_facts')
report = UniverseReport(tuple(survey_from_snapshot(entry, raw_store) for entry in manifest['companies']))
print(f'{len(report.companies)} companies replayed offline from stored SEC snapshots')
for member in UNIVERSE_6A:
    if member.note:
        print(f'  {member.ticker}: {member.note}')

24 companies replayed offline from stored SEC snapshots
  CAT: stress test: captive finance arm (Cat Financial)
  DE: stress test: captive finance arm (John Deere Financial)
  UNH: stress test: insurance-like balance sheet


## 1. Company view

Each row shows the six analytical layers (`OK` available, `PART` partial, `INSUF` insufficient data, `UNAV` unsupported input, `BLOCK` invalid input, `ERR` unexpected exception) and the first blocking metric with its diagnosis.

`FULL` requires every layer to be available **and** no silently stale metric.

In [2]:
print(format_company_view(report))

Ticker Overall  Owner CapEf EcVal Comp  CapAl Summ   Unsupported                        Invalid                  Stale (silent)               Primary blocker
ADBE   FULL     OK    OK    OK    OK    OK    OK     -                                  -                        -                            -
V      PARTIAL  PART  OK    INSUF INSUF PART  INSUF  diluted_shares                     -                        -                            diluted_shares [MISSING_CONCEPT]
COST   FULL     OK    OK    OK    OK    OK    OK     -                                  -                        -                            -
MSFT   FULL     OK    OK    OK    OK    OK    OK     -                                  -                        -                            -
CRM    PARTIAL  OK    UNAV  UNAV  UNAV  UNAV  UNAV   current_debt                       long_term_debt           -                            current_debt [ALTERNATIVE_CONCEPT]
NOW    FAILED   BLOCK BLOCK BLOCK BLOCK BLOCK BLOCK  curre

## 2. Metric view: which canonical metrics block coverage?

For each canonical metric: how many companies have it available, unsupported, invalid (ambiguous or malformed), or structurally absent; how many companies it blocks; and the dominant diagnosis.

In [3]:
print(format_metric_view(report))

Metric                    Avail Unsup Inval Absent Blocks  Dominant diagnosis   Companies affected
revenue                      22     1     1      0      2  AMBIGUOUS_DUPLICATE  unsupported: XOM; invalid: PFE
operating_income             19     5     0      0      5  MISSING_CONCEPT      unsupported: NKE,DE,PFE,XOM,CVX
net_income                   21     2     1      0      2  AMBIGUOUS_DUPLICATE  unsupported: CAT,XOM; invalid: NOW
operating_cash_flow          23     1     0      0      0  PERIOD_ISSUE         unsupported: XOM
capital_expenditures         22     1     1      0      1  AMBIGUOUS_DUPLICATE  unsupported: XOM; invalid: META
diluted_shares               18     2     4      0      5  AMBIGUOUS_DUPLICATE  unsupported: V,XOM; invalid: NOW,AMZN,NVDA,CMG
income_tax_expense           23     1     0      0      0  PERIOD_ISSUE         unsupported: XOM
pretax_income                22     2     0      0      1  STALE_CONCEPT        unsupported: ORCL,XOM
repurchases                 

## 3. Patterns: recurring taxonomy versus company quirks

In [4]:
print(format_pattern_view(report))

Top blocking metrics (companies blocked):
  current_debt              8
  diluted_shares            5
  operating_income          5
  cash                      2
  net_income                2
Diagnosis categories (company-metric findings):
  ALTERNATIVE_CONCEPT    9
  AMBIGUOUS_DUPLICATE    13
  BUSINESS_STRUCTURE     1
  COMPOSITE_CANDIDATE    5
  MISSING_CONCEPT        16
  PERIOD_ISSUE           13
  STALE_CONCEPT          14
Silently stale AVAILABLE metrics (selected concept has no recent value): 0 in 0 companies
Ambiguous duplicates by restatement kind:
  PRECISION       8  CRM:long_term_debt, NOW:net_income, NOW:cash, NOW:total_assets, NOW:total_equity, META:capital_expenditures, INTU:long_term_debt, PFE:cash
  STOCK_SPLIT     4  NOW:diluted_shares, AMZN:diluted_shares, NVDA:diluted_shares, CMG:diluted_shares
  VALUE_CHANGE    1  PFE:revenue
Composite-metric candidates:
  COST   current_debt     value-level: LongTermDebtCurrent excludes non-zero FY2022 OtherShortTermBorrowings 88

## 4. The most important finding: silent staleness (fixed in Slice 6B)

In the 6A survey the SEC selector took the first preferred concept with *any* annual history. When a company stopped tagging that concept, OwnerLens kept the old values and marked the metric `AVAILABLE`. 14 metrics in 10 companies were affected, for example NVIDIA revenue (ending FY2022), Amazon capital expenditures (ending FY2016), and Caterpillar net income (ending FY2010).

Slice 6B made concept selection recency-aware. A concept is eligible only if it covers the company's latest fiscal year, so stale history can no longer appear as `AVAILABLE`. A current fallback concept is selected when one exists in the preference list; otherwise the metric is `UNSUPPORTED` and diagnosed `STALE_CONCEPT`. The cell below lists any silently stale metric. After 6B it should print nothing.

In [5]:
for company in report.surveyed:
    for metric in company.silently_stale():
        print(f'{company.ticker:<5} {metric:<24} {company.metrics[metric].reason[:120]}')

## 5. What would the next normalization improvement unblock?

`unblocked_by` is a what-if over the diagnoses. It counts companies whose every unsupported or invalid required input would be resolved by an improvement. It does not recompute anything, and it ignores silent staleness, so it measures blocked coverage, not trustworthiness.

In [6]:
current_debt = lambda t, f: f.metric == 'current_debt' and f.diagnosis in (D.ALTERNATIVE_CONCEPT, D.COMPOSITE_CANDIDATE, D.MISSING_CONCEPT)
ambiguity = lambda t, f: f.diagnosis is D.AMBIGUOUS_DUPLICATE and f.restatement in (R.PRECISION, R.STOCK_SPLIT)
fallback = lambda t, f: f.diagnosis in (D.ALTERNATIVE_CONCEPT, D.STALE_CONCEPT) and bool(f.candidates)
long_term_debt = lambda t, f: f.metric == 'long_term_debt' and f.diagnosis is not D.PERIOD_ISSUE

scenarios = {
    'A. current-debt policy (fallbacks + composite + absence)': current_debt,
    'B. restatement-aware ambiguity (precision + stock split)': ambiguity,
    'C. recency-aware concept fallback': fallback,
    'A + B': lambda t, f: current_debt(t, f) or ambiguity(t, f),
    'A + B + C': lambda t, f: current_debt(t, f) or ambiguity(t, f) or fallback(t, f),
    'A + B + C + long-term debt': lambda t, f: current_debt(t, f) or ambiguity(t, f) or fallback(t, f) or long_term_debt(t, f),
}
blocked = [c.ticker for c in report.surveyed if c.unusable_inputs()]
print(f'{len(blocked)} of {len(report.companies)} companies have at least one blocking input\n')
for name, resolves in scenarios.items():
    unblocked = report.unblocked_by(resolves)
    print(f'{name:<60} unblocks {len(unblocked):>2}: {", ".join(unblocked)}')

20 of 24 companies have at least one blocking input

A. current-debt policy (fallbacks + composite + absence)     unblocks  2: HD, LOW
B. restatement-aware ambiguity (precision + stock split)     unblocks  1: NVDA
C. recency-aware concept fallback                            unblocks  5: PG, HD, LOW, CAT, UNH
A + B                                                        unblocks  6: CRM, META, NVDA, CMG, HD, LOW
A + B + C                                                    unblocks 11: CRM, META, NVDA, INTU, CMG, PG, KO, HD, LOW, CAT, UNH
A + B + C + long-term debt                                   unblocks 12: CRM, META, NVDA, INTU, LULU, CMG, PG, KO, HD, LOW, CAT, UNH


## 6. Acceptance answers

In [7]:
total = len(report.companies)
dist = report.distribution()
print('1-2. Coverage:', ', '.join(f'{k} {v}/{total} ({v / total:.0%})' for k, v in dist.items()))
print('3.   Top blocking metrics:', ', '.join(f'{m} ({n})' for m, n in report.top_blockers()))
recurring = [r for r in report.recurring_candidates() if r[3]]
print(f'4.   Recurring alternative concepts (3+ companies): {len(recurring)};',
      'quirks:', len(report.recurring_candidates()) - len(recurring))
composites = report.composites()
print(f'5.   Composite candidates: {len(composites)} in {len({c[0] for c in composites})} companies',
      f'({sum(1 for c in composites if c[2])} value-level understatements)')
stale = [c for c in report.surveyed if c.silently_stale()]
print(f'     Silently stale metrics: {sum(len(c.silently_stale()) for c in stale)} in {len(stale)} companies')
print('     Diagnosis mix:', report.diagnosis_counts())

1-2. Coverage: FULL 3/24 (12%), PARTIAL 10/24 (42%), FAILED 11/24 (46%)
3.   Top blocking metrics: current_debt (8), diluted_shares (5), operating_income (5), cash (2), net_income (2)
4.   Recurring alternative concepts (3+ companies): 6; quirks: 6
5.   Composite candidates: 5 in 5 companies (2 value-level understatements)
     Silently stale metrics: 0 in 0 companies
     Diagnosis mix: {'ALTERNATIVE_CONCEPT': 9, 'AMBIGUOUS_DUPLICATE': 13, 'BUSINESS_STRUCTURE': 1, 'COMPOSITE_CANDIDATE': 5, 'MISSING_CONCEPT': 16, 'PERIOD_ISSUE': 13, 'STALE_CONCEPT': 14}


**Reading the answers**

The prose below records the original **6A** survey (before Slice 6B). The cells above replay the stored snapshots with the *current* code, so after 6B they show the recency-aware results. The before and after comparison is in the Feature 6 documentation.

* **Coverage (6A):** 3 of 24 companies (12%) reached full coverage, 12 (50%) were partial, and 9 (38%) failed. A failure means owner economics itself is unavailable. After 6B: 3 full, 10 partial, 11 failed. CAT and DE now fail honestly, because their net income and operating income were stale or discontinued concepts.
* **Top blockers:** current debt dominates. Next come diluted shares (all stock-split or precision restatements) and operating income (energy, pharma, Nike, and after 6B Deere).
* **Recurring versus quirk:** failures are **mostly recurring taxonomy patterns**. Three current-debt alternatives each recur in 3–4 companies. Every ambiguous duplicate is a precision re-rounding or a stock-split restatement, except two genuine revisions. Staleness recurred across 10 companies.
* **Composite metrics:** current debt is composite in 5 companies (3 unsupported, 2 value-level understatements, including MSFT FY2024 commercial paper). No other metric needed a composite.
* **SEC-first at 100+:** practical, *if* the next improvements are general rules rather than per-ticker overrides. The blockers cluster into a handful of policies, and request volume is two SEC calls per company.
* **Next improvement:** recency-aware selection was delivered in 6B. The Feature 6 documentation ranks the remaining 6C candidates: current-debt policy, restatement-aware ambiguity, and catalog concept policy.